In [ ]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


In [ ]:
N_ITERS = 5
T_OFFSET = 69.8
T_DURATION = 90

T_REACH_OFFSET = 70
TARGET_DISCOVERY_RATE = 99

In [ ]:
def read_reachability(target: str, n_peers: int) -> list[list[tuple[float, float]]]:
    data = []
    for i in range(N_ITERS):
        with open(f'../continuous/{target}/{n_peers}/r_{i}.txt', 'r') as f:
            time_init = 0
            measurements = []
            
            for line in f:
                parts = line.strip().split()
                if parts[1] == 'N/A':
                    continue

                time = int(parts[0])
                if time_init == 0:
                    time_init = time
                execution_time = (time - time_init) / 1000 - T_REACH_OFFSET
                reachability = float(parts[1])

                if execution_time < 0:
                    continue
                
                measurements.append((execution_time, reachability * 100))
            data.append(measurements)
    return data

In [ ]:
def time_split_discovery_time(data: list[tuple[float, float]]) -> list[float]:
    result = []
    time_now = 0.1
    split_discovery_time = 0.0
    for time, reach in data:
        if time >= time_now + 10:
            time_now += 10
            if split_discovery_time == 0.0:
                result.append(10.0)
            else:
                result.append(split_discovery_time)
                
            split_discovery_time = 0.0
            continue

        if reach > TARGET_DISCOVERY_RATE and split_discovery_time == 0.0:
            split_discovery_time = time - time_now + 0.1
    return result

In [ ]:
def time_split_discovery_time_all_seeds(target: str, n_peers: int) -> list[float]:
    result = []
    for data in read_reachability(target, n_peers):
        discovery_times = time_split_discovery_time(data)
        result.extend(discovery_times)
    return result

In [ ]:
def read_ifstat_timesum(target: str, n_peers: int) -> list[list[float]]:
    result = [] # on each seed, time_series total bandwidth usage. Time data omitted.
    for seed in range(N_ITERS):
        file_paths = [f'../continuous/{target}/{n_peers}/{seed}/ifstat_h{i+1}.log' for i in range(n_peers)]
        
        files_data = [] # list[list[float]]
        for file_path in file_paths:
            netusages = [] # on time-series, the host's bandwidth usage (kbps)
            with open(file_path, 'r') as f:
                # Skip first two title lines
                next(f, None)
                next(f, None)
                for line_no, line in enumerate(f):
                    execution_time = line_no / 10 - T_OFFSET
                    netusage = sum([float(v) for v in line.strip().split()])
                    
                    # skip before burst
                    if execution_time < 0.0:
                        continue

                    if execution_time > T_DURATION:
                        break
                    
                    netusages.append(netusage)
            files_data.append(netusages)

        combined_netuse = [] # on this run, time-series, total bandwidth usage.
        min_length = min([len(d) for d in files_data])
        for i in range(min_length):
            combined_netuse.append(sum(
                [file_data[i] for file_data in files_data]
            ))

        result.append(combined_netuse)
    return result

In [ ]:
def time_split_interval_netuse(banwidth_time_series: list[float]) -> list[float]:
    result = []
    for i in range(len(banwidth_time_series) // (10 * 10)):
        result.append(sum(
            [v * 0.1 for v in banwidth_time_series[10*10*i:10*10*(i+1)]]
        ) / 8 / 1000) # MB
    return result

In [ ]:
def time_split_interval_netuse_all_seeds(target: str, n_peers) -> list[float]:
    seeds_data = read_ifstat_timesum(target, n_peers)
    
    result = [] # total network usage on a time split, for every seed, every split.
    for seed_data in seeds_data:
        seed_interval_netuses = time_split_interval_netuse(seed_data)
        result.extend(seed_interval_netuses)
    return result

In [ ]:
def draw_dual_grouped_boxplot(
    data_left: list[list[float]],
    data_right: list[list[float]],
    group_labels: list[str],
    legend_labels: list[str] = ("Item 1", "Item 2", "Item 3"),
    y_labels: tuple[str, str] = ("Value A", "Value B"),
    y_ranges: tuple[tuple, tuple] = (None, None),
    colors: list[str] = ["#4C72B0", "#DD8452", "#55A868"],
    figsize: tuple = (12, 5),
):
    n_groups, n_per_group = 3, 3
    group_gap, box_spacing = 1.5, 1.0

    positions, group_centers = [], []
    for g in range(n_groups):
        start = g * (n_per_group * box_spacing + group_gap)
        grp = [start + i * box_spacing for i in range(n_per_group)]
        positions.extend(grp)
        group_centers.append(np.mean(grp))

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    lpads = [1, 1]
    for ax, data, y_label, y_range, lpad in zip(axes, [data_left, data_right], y_labels, y_ranges, lpads):
        bp = ax.boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                        medianprops=dict(color="black", linewidth=1.5))
        for i, patch in enumerate(bp["boxes"]):
            patch.set_facecolor(colors[i % n_per_group])
            patch.set_alpha(0.8)
        ax.set_xticks(group_centers)
        ax.set_xticklabels(group_labels)
        ax.set_ylabel(y_label, labelpad=lpad)
        if y_range is not None:
            ax.set_ylim(y_range)
        ax.grid(True, axis='y', alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)

    handles = [mpatches.Patch(facecolor=colors[i], alpha=0.8, label=legend_labels[i])
               for i in range(n_per_group)]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.06, 0.96), frameon=False)

    plt.tight_layout()
    return fig, axes

In [ ]:
TARGETS = ['client-server', 'dev-eval-trickle', 'dev-v2']
N_PEERS = [110, 130, 200]

rng = np.random.default_rng(42)
fig, axes = draw_dual_grouped_boxplot(
    data_left=[
        time_split_discovery_time_all_seeds(target, n_peers)
        for n_peers in N_PEERS
        for target in TARGETS
    ],
    data_right=[
        time_split_interval_netuse_all_seeds(target, n_peers) 
        for n_peers in N_PEERS
        for target in TARGETS
    ],
    group_labels=['1%', '3%', '10%'],
    legend_labels=['Client-Server', 'Trickle', 'Ours'],
    y_labels=(">99% Peer Discovery Time (s)", "Collective Network Usage (MB)"),
)

plt.savefig("continuous.jpg", dpi=300, bbox_inches="tight")